<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">

## Procesamiento del Lenguaje Natural 1
### Carrera de Especialización en Inteligencia Artificial - FIUBA

## Desafío N° 4
### LSTM Bot QA

### 1º Bimestre 2026

### Grupo

| Autores                     | E-mail                    | Nº SIU  |
|---------------------------- |---------------------------|---------|
| Juan E. Ramos Nervi         | jern10@gmail.com          | a1821   |


### Datos
El objecto es utilizar datos disponibles del challenge ConvAI2 (Conversational Intelligence Challenge 2) de conversaciones en inglés. Se construirá un BOT para responder a preguntas del usuario (QA).\
[LINK](http://convai.io/data/)

In [107]:
!pip install --upgrade --no-cache-dir gdown --quiet
!pip install nltk --quiet


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [108]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jeramosnervi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [109]:
import numpy as np                  
import matplotlib.pyplot as plt     
import seaborn as sns               
import os                           

import nltk
import re

import tensorflow as tf
from tensorflow import keras

In [110]:
# Descargar la carpeta de dataset
import gdown
if os.access('data_volunteers.json', os.F_OK) is False:
    url = 'https://drive.google.com/uc?id=1awUxYwImF84MIT5-jCaYAPe2QwSgS1hN&export=download'
    output = 'data_volunteers.json'
    gdown.download(url, output, quiet=False)
else:
    print("El dataset ya se encuentra descargado")

El dataset ya se encuentra descargado


In [111]:
# dataset_file
import json

text_file = "data_volunteers.json"
with open(text_file) as f:
    data = json.load(f) # la variable data será un diccionario

In [112]:
# Observar los campos disponibles en cada linea del dataset
data[0].keys()



dict_keys(['dialog', 'start_time', 'end_time', 'bot_profile', 'user_profile', 'eval_score', 'profile_match', 'participant1_id', 'participant2_id'])

In [113]:
chat_in = []
chat_out = []

input_sentences = []
output_sentences = []
output_sentences_inputs = []

max_len = 30  # umbral simple para evitar frases demasiado largas

def clean_text(txt):
    txt = txt.lower()

    # normalizaciones básicas
    txt = txt.replace("\'d", " had")
    txt = txt.replace("\'s", " is")
    txt = txt.replace("\'m", " am")
    txt = txt.replace("don't", "do not")

    # deja solo letras/números/espacios
    txt = re.sub(r'\W+', ' ', txt)

    # elimina espacios sobrantes
    txt = txt.strip()

    return txt

for line in data:
    # recorre pares consecutivos dentro de cada diálogo
    for i in range(len(line['dialog']) - 1):

        # turno actual = entrada
        chat_in = clean_text(line['dialog'][i]['text'])

        # turno siguiente = salida
        chat_out = clean_text(line['dialog'][i + 1]['text'])

        # conviene filtrar vacíos
        if len(chat_in) == 0 or len(chat_out) == 0:
            continue

        # filtro simple por longitud de caracteres
        #if len(chat_in) >= max_len or len(chat_out) >= max_len:
        #    continue
        # filtro simple por longitud de palabras 
        if len(chat_in.split()) >= max_len or len(chat_out.split()) >= max_len:
            continue
        
        input_sentence = chat_in
        output = chat_out

        # salida esperada del decoder
        output_sentence = output + ' <eos>'

        # entrada al decoder
        output_sentence_input = '<sos> ' + output

        input_sentences.append(input_sentence)
        output_sentences.append(output_sentence)
        output_sentences_inputs.append(output_sentence_input)

print("Cantidad de filas utilizadas:", len(input_sentences))

Cantidad de filas utilizadas: 13186


In [114]:
input_sentences[1], output_sentences[1], output_sentences_inputs[1]

('hi how are you', 'not bad and you <eos>', '<sos> not bad and you')

### 2 - Preprocesamiento

Se realiza el preprocesamiento del corpus para preparar los datos del modelo encoder–decoder.

Se obtienen:
- word2idx_inputs, max_input_len
  Vocabulario y longitud máxima de las entradas.

- word2idx_outputs, max_out_len, num_words_output  
  Vocabulario, longitud máxima y tamaño del vocabulario de salida.

- encoder_input_sequences, decoder_output_sequences, decoder_targets 
  Secuencias numéricas listas para entrenamiento:
  - encoder inputs  
  - decoder inputs (`<sos>`)  
  - targets (`<eos>`)

In [115]:
from tensorflow.keras.preprocessing.text import Tokenizer

# Tamaño máximo del vocabulario
MAX_VOCAB_SIZE = 3000

# Filtros:
# - quitamos signos comunes
# - dejamos < y > para conservar <sos> y <eos>
# - agregamos ¿ y sacamos algunos símbolos que no queremos
filters_custom = '!"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n¿'

# Tokenizador para inputs
input_tokenizer = Tokenizer(
    num_words=MAX_VOCAB_SIZE,
    filters=filters_custom,
    oov_token="<unk>"
)

input_tokenizer.fit_on_texts(input_sentences)
input_integer_seq = input_tokenizer.texts_to_sequences(input_sentences)

word2idx_inputs = input_tokenizer.word_index

# cantidad efectiva de palabras que va a usar el tokenizador
num_words_input = min(MAX_VOCAB_SIZE, len(word2idx_inputs) + 1)

print("Palabras detectadas en input:", len(word2idx_inputs))
print("Tamaño efectivo del vocabulario input:", num_words_input)

max_input_len = max(len(seq) for seq in input_integer_seq)
print("oracion de entrada más larga:", max_input_len)

Palabras detectadas en input: 3862
Tamaño efectivo del vocabulario input: 3000
oracion de entrada más larga: 29


In [116]:
print("Primeros 10 tokens:", list(word2idx_inputs.items())[:10])

Primeros 10 tokens: [('<unk>', 1), ('i', 2), ('you', 3), ('do', 4), ('a', 5), ('what', 6), ('am', 7), ('to', 8), ('is', 9), ('like', 10)]


#### 2.1 - Tokenización
Aplicamos el tokenizador de keras para convertir el texto en una secuencia de palabras.

In [117]:
from tensorflow.keras.preprocessing.text import Tokenizer

MAX_VOCAB_SIZE = 8000

# Filtros (no sacamos < > para mantener <sos> y <eos>)
filters_custom = '!"#$%&()*+,-./:;=¿?@[\\]^_`{|}~\t\n'

input_tokenizer = Tokenizer(
    num_words=MAX_VOCAB_SIZE,
    filters=filters_custom,
    oov_token="<unk>"
)

input_tokenizer.fit_on_texts(input_sentences)
input_integer_seq = input_tokenizer.texts_to_sequences(input_sentences)

word2idx_inputs = input_tokenizer.word_index

# tamaño efectivo del vocabulario
num_words_input = min(MAX_VOCAB_SIZE, len(word2idx_inputs) + 1)

print("Palabras detectadas:", len(word2idx_inputs))
print("Tamaño efectivo vocabulario:", num_words_input)

# longitud máxima
max_input_len = max(len(seq) for seq in input_integer_seq)
print("oracion de entrada más larga:", max_input_len)

Palabras detectadas: 3862
Tamaño efectivo vocabulario: 3863
oracion de entrada más larga: 29


In [118]:
output_tokenizer = Tokenizer(
    num_words=MAX_VOCAB_SIZE,
    filters=filters_custom,
    oov_token="<unk>"
)

# importante incluir <sos> y <eos> en el vocabulario
output_tokenizer.fit_on_texts(["<sos>", "<eos>"] + output_sentences)

output_integer_seq = output_tokenizer.texts_to_sequences(output_sentences)
output_input_integer_seq = output_tokenizer.texts_to_sequences(output_sentences_inputs)

word2idx_outputs = output_tokenizer.word_index

print("Palabras detectadas:", len(word2idx_outputs))

# tamaño efectivo del vocabulario
num_words_output = min(MAX_VOCAB_SIZE, len(word2idx_outputs) + 1)

print("Tamaño efectivo vocabulario salida:", num_words_output)

# longitud máxima de salida
max_out_len = max(len(seq) for seq in output_integer_seq)
print("Sentencia de salida más larga:", max_out_len)

Palabras detectadas: 3843
Tamaño efectivo vocabulario salida: 3844
Sentencia de salida más larga: 30


In [119]:
print("<sos> index:", word2idx_outputs.get("<sos>"))
print("<eos> index:", word2idx_outputs.get("<eos>"))

<sos> index: 1980
<eos> index: 2


In [120]:
print(output_sentences[0])
print(output_integer_seq[0])

print(output_sentences_inputs[0])
print(output_input_integer_seq[0])

hi how are you <eos>
[41, 15, 12, 4, 2]
<sos> hi how are you
[1980, 41, 15, 12, 4]


In [121]:
# Se fija una longitud máxima manual para limitar consumo de memoria
# y mantener tamaños de secuencia razonables para entrenamiento.
max_input_len = max(len(seq) for seq in input_integer_seq)
max_out_len = max(len(seq) for seq in output_integer_seq)

print("max_input_len final:", max_input_len)
print("max_out_len final:", max_out_len)

max_input_len final: 29
max_out_len final: 30


In [122]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("Cantidad de instancias del dataset:", len(input_integer_seq))

encoder_input_sequences = pad_sequences(
    input_integer_seq,
    maxlen=max_input_len,
    padding='post'
)
print("encoder_input_sequences shape:", encoder_input_sequences.shape)

decoder_input_sequences = pad_sequences(
    output_input_integer_seq,
    maxlen=max_out_len,
    padding='post'
)
print("decoder_input_sequences shape:", decoder_input_sequences.shape)
print(input_integer_seq[0])
print(encoder_input_sequences[0])

Cantidad de instancias del dataset: 13186
encoder_input_sequences shape: (13186, 29)
decoder_input_sequences shape: (13186, 30)
[36]
[36  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0]


Se aplica padding sobre las secuencias de entrada y salida para unificar sus longitudes. En ambos casos se utiliza `padding='post'`, de modo que los ceros se agregan al final de cada secuencia. Esto permite mantener una representación consistente para el modelo.

In [123]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Padding de las secuencias de salida (targets reales)
decoder_output_sequences = pad_sequences(
    output_integer_seq,
    maxlen=max_out_len,
    padding='post'
)

# En lugar de usar one-hot (to_categorical), usamos directamente enteros
# Esto permite usar sparse_categorical_crossentropy y reduce mucho el consumo de memoria
decoder_targets = decoder_output_sequences

print("decoder_targets shape:", decoder_targets.shape)

decoder_targets shape: (13186, 30)


### 3 - Preparar los embeddings
Utilizar los embeddings de Glove o FastText para transformar los tokens de entrada en vectores

In [124]:
import os
import gdown

# ===== GloVe =====
if not os.path.exists("gloveembedding.pkl"):
    try:
        gdown.download(
            id="1KY6avD5I1eI2dxQzMkR3WExwKwRq2g94",
            output="gloveembedding.pkl",
            quiet=False
        )
    except Exception as e:
        print("No se pudo descargar GloVe")
        print("Detalle:", e)
else:
    print("Los embeddings gloveembedding.pkl ya están descargados")

# ===== FastText =====
if not os.path.exists("fasttext.pkl"):
    try:
        # probamos con el ID directo (más robusto que URL ya que me fallo antes)
        gdown.download(
            id="1Qi1r-u5lsEsNqRSxLrpN0qQ3B_ufltCa",
            output="fasttext.pkl",
            quiet=False
        )
    except Exception as e:
        print("No se pudo descargar fasttext.pkl con gdown")
        print("Se continuará usando GloVe")
        print("Detalle:", e)
else:
    print("Los embeddings fasttext.pkl ya están descargados")

Los embeddings gloveembedding.pkl ya están descargados
No se pudo descargar fasttext.pkl con gdown
Se continuará usando GloVe
Detalle: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1Qi1r-u5lsEsNqRSxLrpN0qQ3B_ufltCa

but Gdown can't. Please check connections and permissions.


In [125]:
import os
import pickle

# ===== GloVe =====
with open("gloveembedding.pkl", "rb") as f:
    glove = pickle.load(f)

print("Cantidad de palabras en GloVe:", len(glove))

# ===== FastText (opcional) =====
if os.path.exists("fasttext.pkl"):
    with open("fasttext.pkl", "rb") as f:
        fasttext = pickle.load(f)
    print("Cantidad de palabras en FastText:", len(fasttext))
else:
    fasttext = None
    print("FastText no disponible, se usará solo GloVe")

Cantidad de palabras en GloVe: 1193514
FastText no disponible, se usará solo GloVe


C:\Users\jeramosnervi\AppData\Local\Temp\ipykernel_20972\1433634625.py:6: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  glove = pickle.load(f)


In [126]:
print("Ejemplo GloVe:")
for i in range(3):
    word = glove['word'][i]
    vector = glove['embedding'][i]
    print(word, vector[:5])

# ===== FastText =====
if fasttext is not None:
    print("\nEjemplo FastText:")
    for i in range(3):
        word = fasttext['word'][i]
        vector = fasttext['embedding'][i]
        print(word, vector[:5])
else:
    print("\nFastText no disponible")

Ejemplo GloVe:
<user> [ 0.78704   0.72151   0.29148  -0.056527  0.31683 ]
. [ 0.68661  -1.0772    0.011114 -0.24075  -0.3422  ]
: [0.98483 0.19784 0.28403 0.35406 0.2438 ]

FastText no disponible


sigo solo con GloVe

In [127]:
import logging
import os
from pathlib import Path
import pickle
import numpy as np

class WordsEmbeddings:
    logger = logging.getLogger(__name__)

    def __init__(self):
        words_embedding_pkl = Path(self.PKL_PATH)

        # si existe el pickle, lo carga
        if words_embedding_pkl.is_file():
            embeddings = self.load_model_from_pickle()
        else:
            # si no existe el pickle pero sí el txt, convierte
            words_embedding_txt = Path(self.WORD_TO_VEC_MODEL_TXT_PATH)
            assert words_embedding_txt.is_file(), f'Words embedding not available: {self.WORD_TO_VEC_MODEL_TXT_PATH}'
            embeddings = self.convert_model_to_pickle()

        self.embeddings = embeddings

        # diccionarios palabra <-> índice
        index = np.arange(self.embeddings.shape[0])
        self.word2idx = dict(zip(self.embeddings['word'], index))
        self.idx2word = dict(zip(index, self.embeddings['word']))

        # índice del embedding nulo
        self.null_idx = self.word2idx['null_embedding']

    def get_words_embeddings(self, words):
        words_idxs = self.words2idxs(words)
        return self.embeddings[words_idxs]['embedding']

    def words2idxs(self, words):
        return np.array([self.word2idx.get(word, self.null_idx) for word in words])

    def idxs2words(self, idxs):
        return np.array([self.idx2word.get(idx, 'null_embedding') for idx in idxs])

    def load_model_from_pickle(self):
        self.logger.debug(f'loading words embeddings from pickle {self.PKL_PATH}')

        max_bytes = 2**28 - 1  # 256MB
        bytes_in = bytearray(0)
        input_size = os.path.getsize(self.PKL_PATH)

        with open(self.PKL_PATH, 'rb') as f_in:
            for _ in range(0, input_size, max_bytes):
                bytes_in += f_in.read(max_bytes)

        embeddings = pickle.loads(bytes_in)
        self.logger.debug('words embeddings loaded')
        return embeddings

    def convert_model_to_pickle(self):
        self.logger.debug(
            f'converting and loading words embeddings from text file {self.WORD_TO_VEC_MODEL_TXT_PATH}'
        )

        structure = np.dtype([
            ('word', np.dtype('U' + str(self.WORD_MAX_SIZE))),
            ('embedding', np.float32, (self.N_FEATURES,))
        ])

        with open(self.WORD_TO_VEC_MODEL_TXT_PATH, encoding="utf8") as words_embeddings_txt:
            embeddings_gen = (
                (line.split()[0], line.split()[1:])
                for line in words_embeddings_txt
                if len(line.split()[1:]) == self.N_FEATURES
            )
            embeddings = np.fromiter(embeddings_gen, dtype=structure)

        # agregar embedding nulo
        null_embedding = np.array(
            [('null_embedding', np.zeros((self.N_FEATURES,), dtype=np.float32))],
            dtype=structure
        )
        embeddings = np.concatenate([embeddings, null_embedding])

        # guardar como pickle
        max_bytes = 2**28 - 1
        bytes_out = pickle.dumps(embeddings, protocol=pickle.HIGHEST_PROTOCOL)

        with open(self.PKL_PATH, 'wb') as f_out:
            for idx in range(0, len(bytes_out), max_bytes):
                f_out.write(bytes_out[idx:idx + max_bytes])

        self.logger.debug('words embeddings converted and saved')
        return embeddings


class GloveEmbeddings(WordsEmbeddings):
    # si solo tenés el pickle, no hace falta el txt
    WORD_TO_VEC_MODEL_TXT_PATH = 'glove.twitter.27B.50d.txt'
    PKL_PATH = 'gloveembedding.pkl'
    N_FEATURES = 50
    WORD_MAX_SIZE = 60


class FasttextEmbeddings(WordsEmbeddings):
    # solo funcionará si alguna vez tengo el archivo
    WORD_TO_VEC_MODEL_TXT_PATH = 'cc.en.300.vec'
    PKL_PATH = 'fasttext.pkl'
    N_FEATURES = 300
    WORD_MAX_SIZE = 60

In [128]:
model_embeddings = GloveEmbeddings()

C:\Users\jeramosnervi\AppData\Local\Temp\ipykernel_20972\2091071747.py:53: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  embeddings = pickle.loads(bytes_in)


### 4 - Entrenar el modelo
Entrenar un modelo basado en el esquema encoder-decoder.

### 4 -1 Arquitectura

In [129]:
import os
import random

rnd_seed = 42

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(rnd_seed)

In [130]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding

# hiperparámetros
RNN_UNITS = 256
DROPOUT = 0.1

# =========================
# Encoder
# =========================
encoder_inputs = Input(shape=(max_input_len,), name="encoder_input")

# si existe embedding_matrix usa GloVe, sino embedding random
if 'embedding_matrix' in globals():
    print("Usando embeddings preentrenados (GloVe)")
    encoder_embedding_layer = Embedding(
        input_dim=num_words_input,
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=False,
        mask_zero=True,
        name="encoder_embedding"
    )
else:
    print("Usando embeddings entrenables (random)")
    encoder_embedding_layer = Embedding(
        input_dim=num_words_input,
        output_dim=128,
        mask_zero=True,
        name="encoder_embedding"
    )

encoder_inputs_x = encoder_embedding_layer(encoder_inputs)

encoder_lstm = LSTM(
    RNN_UNITS,
    return_state=True,
    dropout=DROPOUT,
    name="encoder_lstm"
)

_, state_h, state_c = encoder_lstm(encoder_inputs_x)
encoder_states = [state_h, state_c]

# =========================
# Decoder
# =========================
decoder_inputs = Input(shape=(max_out_len,), name="decoder_input")

decoder_embedding_layer = Embedding(
    input_dim=num_words_output,
    output_dim=RNN_UNITS,
    mask_zero=True,
    name="decoder_embedding"
)

decoder_inputs_x = decoder_embedding_layer(decoder_inputs)

decoder_lstm = LSTM(
    RNN_UNITS,
    return_sequences=True,
    return_state=True,
    dropout=DROPOUT,
    name="decoder_lstm"
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_inputs_x,
    initial_state=encoder_states
)

# =========================
# Salida
# =========================
decoder_dense = Dense(
    num_words_output,
    activation='softmax',
    name="decoder_output_dense"
)

decoder_outputs = decoder_dense(decoder_outputs)

# =========================
# Modelo final
# =========================
model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs,
    name="seq2seq_model"
)

model.summary()

Usando embeddings entrenables (random)


Model: "seq2seq_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input       │ (None, 29)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_input       │ (None, 30)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 29, 128)   │    494,464 │ encoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_13        │ (None, 29)        │          0 │ encoder_input[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 30, 256)   │    984,064 │ decoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 256),     │    394,240 │ encoder_embeddin… │
│                     │ (None, 256),      │            │ not_equal_13[0][… │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 30, 256), │    525,312 │ decoder_embeddin… │
│                     │ (None, 256),      │            │ encoder_lstm[0][… │
│                     │ (None, 256)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_output_den… │ (None, 30, 3844)  │    987,908 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,385,988 (12.92 MB)

 Trainable params: 3,385,988 (12.92 MB)

 Non-trainable params: 0 (0.00 B)

Modelo solo ENCODER

In [131]:
# define inference encoder
encoder_model = Model(encoder_inputs, encoder_states)

encoder_model.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input       │ (None, 29)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 29, 128)   │    494,464 │ encoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_13        │ (None, 29)        │          0 │ encoder_input[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 256),     │    394,240 │ encoder_embeddin… │
│                     │ (None, 256),      │            │ not_equal_13[0][… │
│                     │ (None, 256)]      │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 888,704 (3.39 MB)

 Trainable params: 888,704 (3.39 MB)

 Non-trainable params: 0 (0.00 B)

### Modelo solo decoder

In [132]:
# define inference decoder
decoder_state_input_h = Input(shape=(RNN_UNITS,), name="decoder_state_input_h")
decoder_state_input_c = Input(shape=(RNN_UNITS,), name="decoder_state_input_c")
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

# En inferencia, el decoder recibe una sola palabra por vez
decoder_inputs_single = Input(shape=(1,), name="decoder_input_single")
decoder_inputs_single_x = decoder_embedding_layer(decoder_inputs_single)

decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_inputs_single_x,
    initial_state=decoder_states_inputs
)

decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_outputs)

decoder_model = Model(
    [decoder_inputs_single] + decoder_states_inputs,
    [decoder_outputs] + decoder_states
)

decoder_model.summary()

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ decoder_input_sing… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 1, 256)    │    984,064 │ decoder_input_si… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_state_inpu… │ (None, 256)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_state_inpu… │ (None, 256)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 1, 256),  │    525,312 │ decoder_embeddin… │
│                     │ (None, 256),      │            │ decoder_state_in… │
│                     │ (None, 256)]      │            │ decoder_state_in… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_output_den… │ (None, 1, 3844)   │    987,908 │ decoder_lstm[1][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,497,284 (9.53 MB)

 Trainable params: 2,497,284 (9.53 MB)

 Non-trainable params: 0 (0.00 B)

### 4 - 2 Entrenamiento

In [133]:
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import load_model

# hiperparámetros de entrenamiento
EPOCHS = 50
LR = 1e-3

lr_schedule = CosineDecay(
    initial_learning_rate=LR,
    decay_steps=EPOCHS,
    alpha=1e-4
)

#optimizer = Adam(learning_rate=lr_schedule)
optimizer = Adam(learning_rate=1e-3)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    verbose=1,
    restore_best_weights=True
)

# IMPORTANTE:
# decoder_targets contiene enteros, no one-hot
# por eso usamos sparse_categorical_crossentropy
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=optimizer,
    metrics=['accuracy']
)

filename = 'QABot_model_v0.keras'
encoder_file = 'QABot_encoder_v0.keras'
decoder_file = 'QABot_decoder_v0.keras'

if os.path.exists(filename):
    print("Cargando modelo guardado...")
    model = load_model(filename)
else:
    print("Entrenando modelo...")

    history = model.fit(
        [encoder_input_sequences, decoder_input_sequences],
        decoder_targets,
        validation_split=0.20,
        callbacks=[early_stop],
        epochs=EPOCHS,
        verbose=1
    )

    # guardar modelo y encoder/decoder de inferencia
    model.save(filename)
    encoder_model.save(encoder_file)
    decoder_model.save(decoder_file)

    # gráficos de entrenamiento
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.set(style="whitegrid", palette="muted", font_scale=1.2)

    epoch_count = range(1, len(history.history['accuracy']) + 1)

    # Loss
    sns.lineplot(
        x=epoch_count,
        y=history.history['loss'],
        label='Train Loss',
        linewidth=2,
        ax=axes[0]
    )
    sns.lineplot(
        x=epoch_count,
        y=history.history['val_loss'],
        label='Validation Loss',
        linewidth=2,
        ax=axes[0]
    )
    axes[0].set_title('Loss by epochs', fontsize=16)
    axes[0].set_xlabel('Epoch', fontsize=14)
    axes[0].set_ylabel('Loss', fontsize=14)
    axes[0].legend(loc='upper right', fontsize=12)
    axes[0].grid(True)

    # Accuracy
    sns.lineplot(
        x=epoch_count,
        y=history.history['accuracy'],
        label='Train Accuracy',
        linewidth=2,
        ax=axes[1]
    )
    sns.lineplot(
        x=epoch_count,
        y=history.history['val_accuracy'],
        label='Validation Accuracy',
        linewidth=2,
        ax=axes[1]
    )
    axes[1].set_title('Accuracy by epochs', fontsize=16)
    axes[1].set_xlabel('Epoch', fontsize=14)
    axes[1].set_ylabel('Accuracy', fontsize=14)
    axes[1].legend(loc='lower right', fontsize=12)
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()

Cargando modelo guardado...


### 5 - Inferencia
Experimentar el funcionamiento del modelo.

In [94]:
idx2word_input = {v: k for k, v in word2idx_inputs.items()}
idx2word_target = {v: k for k, v in word2idx_outputs.items()}

# fallback seguro
def idx_to_word_target(idx):
    return idx2word_target.get(idx, "<unk>")

In [95]:
def answer_question(input_seq):
    # Obtiene los estados del encoder
    states_value = encoder_model.predict(input_seq, verbose=0)

    # Inicializa la entrada del decoder con <sos>
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = word2idx_outputs['<sos>']

    eos = word2idx_outputs['<eos>']
    output_sentence = []

    for _ in range(max_out_len):
        # Predicción del próximo token
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value,
            verbose=0
        )

        idx = np.argmax(output_tokens[0, 0, :])

        # Fin de secuencia
        if idx == eos:
            break

        # Índice a palabra
        if idx > 0:
            word = idx2word_target.get(idx, "<unk>")

            # evitar tokens especiales en la salida final
            if word not in ["<sos>", "<eos>"]:
                output_sentence.append(word)

        # actualizar estados
        states_value = [h, c]

        # realimentar el decoder con el token predicho
        target_seq[0, 0] = idx

    return ' '.join(output_sentence)

In [96]:
input_test = "What is your name?"
print("Input:", input_test)

integer_seq_test = input_tokenizer.texts_to_sequences([input_test])[0]
print("Representación en vector de ids:", integer_seq_test)

encoder_sequence_test = pad_sequences(
    [integer_seq_test],
    maxlen=max_input_len,
    padding='post'
)
print("Padding del vector:", encoder_sequence_test)

translation = answer_question(encoder_sequence_test)
print("Response:", translation)

Input: What is your name?
Representación en vector de ids: [6, 9, 22, 54]
Padding del vector: [[ 6  9 22 54  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0]]
Response: i am a teacher


In [98]:
input_test = "How old are you?"
print("Input:", input_test)

# aplicar misma limpieza que en entrenamiento
input_test_clean = clean_text(input_test)

integer_seq_test = input_tokenizer.texts_to_sequences([input_test_clean])[0]
print("Representación en vector de ids:", integer_seq_test)

encoder_sequence_test = pad_sequences(
    [integer_seq_test],
    maxlen=max_input_len,
    padding='post'
)
print("Padding del vector:", encoder_sequence_test)

translation = answer_question(encoder_sequence_test)
print("Response:", translation)

Input: How old are you?
Representación en vector de ids: [14, 81, 11, 3]
Padding del vector: [[14 81 11  3  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0]]
Response: i am fine


In [99]:
input_test = "How are you?"
print("Input:", input_test)

input_test_clean = clean_text(input_test)

integer_seq_test = input_tokenizer.texts_to_sequences([input_test_clean])[0]
print("Representación en vector de ids:", integer_seq_test)

encoder_sequence_test = pad_sequences(
    [integer_seq_test],
    maxlen=max_input_len,
    padding='post'
)
print("Padding del vector:", encoder_sequence_test)

translation = answer_question(encoder_sequence_test)
print("Response:", translation)

Input: How are you?
Representación en vector de ids: [14, 11, 3]
Padding del vector: [[14 11  3  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0]]
Response: i am fine


In [100]:
input_test = "Do you like basketball?"
print("Input:", input_test)

# limpieza (clave)
input_test_clean = clean_text(input_test)

integer_seq_test = input_tokenizer.texts_to_sequences([input_test_clean])[0]
print("Representación en vector de ids:", integer_seq_test)

encoder_sequence_test = pad_sequences(
    [integer_seq_test],
    maxlen=max_input_len,
    padding='post'
)
print("Padding del vector:", encoder_sequence_test)

translation = answer_question(encoder_sequence_test)
print("Response:", translation)

Input: Do you like basketball?
Representación en vector de ids: [4, 3, 10, 304]
Padding del vector: [[  4   3  10 304   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0]]
Response: i do not have a lot of time for that


**OBSERVACIONES**

- Se observa que en las primeras dos preguntas, las respuestas no tuvierin coherencia con lo pedido , mientras que para las últimas 2, la respuesta si.

### 6 - Conclusión

- El *accuracy* alcanzado aparenta ser muy bueno, no obstante, la respuestas obtenidas durante la prueba del modelo estuvieron acordes en algunos casos y fallaron en otros, aunque si es de resaltar que gramaticalmente estuvieron en todos los casos bien.